In [2]:
import json
import pandas as pd
import numpy as np

In [5]:
df_6 = pd.read_csv('../data/old_data/sixth_batch_senators.csv')
df_7 = pd.read_csv('../data/old_data/seventh_batch_senators.csv')
df_8 = pd.read_csv('../data/old_data/eighth_batch_senators.csv')

In [6]:
df_6_8 = pd.concat([df_6,df_7,df_8])

In [7]:
import pandas as pd
import json

df = df_6_8


def parse_congresses(val):
    if pd.isna(val) or val == '':
        return []
    try:
        return json.loads(val)
    except:
        return []

df['congresses_parsed'] = df['congresses'].apply(parse_congresses)

# cleaning df
df['num_terms']      = df['congresses_parsed'].apply(len)
df['positions']      = df['congresses_parsed'].apply(lambda x: list({c.get('position') for c in x}))
df['states']         = df['congresses_parsed'].apply(lambda x: list({c.get('stateName') for c in x}))
df['parties']        = df['congresses_parsed'].apply(lambda x: list({p for c in x for p in c.get('parties', [])}))
df['congress_range'] = df['congresses_parsed'].apply(lambda x: f"{min(c.get('congressNumber',0) for c in x)}-{max(c.get('congressNumber',0) for c in x)}" if x else '')

df = df.drop(columns=['congresses', 'congresses_parsed'])


In [8]:
df = df.drop(columns = ['unaccentedGivenName','unaccentedMiddleName','unaccentedFamilyName','nickName', 'honorificPrefix', 'honorificTitle',])

In [9]:
df.to_csv('../data/6_8_senators_wip.csv', index=False)

In [10]:
df_clean = pd.read_csv('../data/6_8_senators_wip.csv')
df_clean.head()

,id,givenName,familyName,middleName,honorificSuffix,birthYear,deathYear,num_terms,positions,states,parties,congress_range
0,A000001,Fred,Aandahl,George,NaN,1897,1966.0,1,['Representative'],['ND'],['Republican'],82-82
1,A000002,Watkins,Abbitt,Moorman,NaN,1908,1998.0,13,['Representative'],['VA'],['Democrat'],80-92
2,A000015,Charles,Abernethy,Laban,NaN,1872,1955.0,7,['Representative'],['NC'],['Democrat'],67-73
3,A000016,Thomas,Abernethy,Gerstle,NaN,1903,1998.0,15,['Representative'],['MS'],['Democrat'],78-92
4,A000021,Ernest,Ackerman,Robinson,NaN,1863,1931.0,7,['Representative'],['NJ'],['Republican'],66-72


In [11]:
df_clean['bio'] = ''

In [12]:
import pandas as pd
import re
import random
from playwright.async_api import async_playwright

df_clean = pd.read_csv('../data/6_8_senators_wip.csv')

# Resume from checkpoint if it exists
try:
    checkpoint = pd.read_csv('senators_checkpoint.csv')
    done_ids = set(checkpoint.loc[checkpoint['bio'].notna() & (checkpoint['bio'] != ''), 'id'])
    df_clean = checkpoint
    print(f"Resuming from checkpoint: {len(done_ids)} already done")
except FileNotFoundError:
    df_clean['bio'] = ''
    done_ids = set()

def extract_bio(body_text):
    # Match text starting with an all-caps last name, e.g. "AANDAHL, Fred George, a Representative..."
    match = re.search(r'([A-Z]{2,}[A-Z\s,]+,\s[A-Z][a-z]+.*?)(?=Content & Site Support)', body_text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return ''

async def scrape():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        for i, row in df_clean.iterrows():
            member_id = row['id']

            if member_id in done_ids:
                continue

            url = f"https://bioguide.congress.gov/search/bio/{member_id}"

            try:
                await page.goto(url, wait_until="networkidle", timeout=30000)
                await page.wait_for_timeout(2000)

                body_text = await page.inner_text("body")
                bio = extract_bio(body_text)

                if bio:
                    print(f"✓ {member_id}")
                else:
                    print(f"~ {member_id}: page loaded but bio not found")

            except Exception as e:
                bio = ''
                print(f"✗ {member_id}: {e}")

            df_clean.at[i, 'bio'] = bio

            # Checkpoint every 50 rows
            if i % 50 == 0:
                df_clean.to_csv('senators_checkpoint.csv', index=False)
                print(f"  [checkpoint saved at row {i}]")

            await page.wait_for_timeout(random.randint(1000, 2500))

        await browser.close()

    df_clean.to_csv('../data/6_8_senators_with_bios.csv', index=False)
    print(f"\nDone.")

await scrape()

Resuming from checkpoint: 4498 already done
✗ C000268: Page.goto: Timeout 30000ms exceeded.
Call log:
  - navigating to "https://bioguide.congress.gov/search/bio/C000268", waiting until "networkidle"

~ C000661: page loaded but bio not found
~ D000090: page loaded but bio not found
~ M000460: page loaded but bio not found
~ P000013: page loaded but bio not found
~ P000385: page loaded but bio not found
~ P000475: page loaded but bio not found
~ B000047: page loaded but bio not found
~ B001091: page loaded but bio not found
~ C000775: page loaded but bio not found
~ C000788: page loaded but bio not found
~ E000284: page loaded but bio not found
~ H001011: page loaded but bio not found
~ L000045: page loaded but bio not found
~ L000506: page loaded but bio not found
~ M000460: page loaded but bio not found
~ M000656: page loaded but bio not found
~ N000110: page loaded but bio not found
~ P000475: page loaded but bio not found
~ R000547: page loaded but bio not found
~ S000394: page load

In [13]:
import pandas as pd
import re

df = pd.read_csv('../data/6_8_senators_with_bios.csv')

def clean_bio(bio):
    if not isinstance(bio, str) or bio.strip() == '':
        return ''
    # Cut off footer
    bio = re.split(r'Content & Site Support', bio)[0]
    # The actual bio sentence always comes after the party name on its own line
    # Split on the last occurrence of a known party label on its own line
    match = re.split(r'\n(?:Republican|Democrat|Whig|Federalist|Independent|Democratic Republican|Crawford Republican|[A-Za-z ]+)\s*\n', bio)
    if match:
        return match[-1].strip()
    return bio.strip()

df['bio'] = df['bio'].apply(clean_bio)

df.to_csv('../data/6_8_senators_with_bios.csv', index=False)
print("Done. Sample output:")
print(df['bio'].head(3).to_string())

Done. Sample output:
0    AANDAHL, Fred George, a Representative from No...
1    ABBITT, Watkins Moorman, a Representative from...
2    ABERNETHY, Charles Laban, a Representative fro...


In [15]:
df_1 = pd.read_csv('../data/senators/first_batch_senators_with_bios.csv')
df_2 = pd.read_csv('../data/senators/second_batch_senators_with_bios.csv')
df_3 = pd.read_csv('../data/senators/3_5_senators_with_bios.csv')
df_4 = pd.read_csv('../data/senators/6_8_senators_with_bios.csv')

df_combined = pd.concat([df_1,df_2,df_3,df_4])


In [17]:
df_combined.head()

,id,givenName,familyName,middleName,honorificSuffix,birthYear,deathYear,num_terms,positions,states,parties,congress_range,bio
0,A000003,Joel,Abbot,NaN,NaN,1776.0,1826.0,4,['Representative'],['GA'],"['Democratic Republican', 'Crawford Republican']",15-18,"Crawford Republican\nABBOT, Joel, a Representa..."
1,A000026,John,Adair,NaN,NaN,1757.0,1840.0,2,"['Senator', 'Representative']",['KY'],"['Democratic Republican', 'Jacksonian']",9-22,"Jacksonian\nADAIR, John, a Senator and a Repre..."
2,A000030,Benjamin,Adams,NaN,NaN,1764.0,1837.0,3,['Representative'],['MA'],['Federalist'],14-16,"ADAMS, Benjamin, a Representative from Massach..."
3,A000041,John,Adams,Quincy,NaN,1767.0,1848.0,12,"['Senator', 'Representative']",['MA'],"['Whig', 'Federalist', 'Anti-Jacksonian', 'Ant...",8-30,"ADAMS, John Quincy, (son of John Adams, father..."
4,A000039,John,Adams,NaN,NaN,1735.0,1826.0,2,['Delegate'],['MA'],[],1-2,"ADAMS, John, (father of John Quincy Adams, gra..."


In [21]:
df_combined = df_combined.sort_values(by="id")
df_combined = df_combined.drop_duplicates()

In [22]:

df_combined.to_csv('../data/all_senators_with_bios.csv', index=False)

In [24]:
df_combined.shape

(13861, 13)